In [26]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
import sys, os
sys.path.append(os.path.abspath(".."))
from braindecode.models import EEGNet
from eeg_preprocessing.loaders import build_loaders

DATASET = "FACED"
EPOCHS=50
LR = 1e-3

CLASSES = {
    "BCI2A": 4,
    "PHYSIONETMI": 4,
    "FACED": 9
}

In [30]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    for x, y in loader:
        y_pred.append(model(x.to(device).float()).argmax(1).cpu())
        y_true.append(y)
    return torch.cat(y_true), torch.cat(y_pred)


def train_eegnet(train_loader, val_loader, n_chans, n_times, n_classes, class_weights=None, **eegnet_kwargs):
    model = EEGNet(n_chans=n_chans, n_outputs=n_classes, n_times=n_times, **eegnet_kwargs).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_loader))
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_bacc, best_state = 0.0, None
    for epoch in range(EPOCHS):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device).float(), y.to(device).long()
            optimizer.zero_grad()
            criterion(model(x), y).backward()
            optimizer.step()
            scheduler.step()
        yt, yp = evaluate(model, val_loader)
        val_bacc = balanced_accuracy_score(yt, yp)
        if val_bacc > best_bacc:
            best_bacc = val_bacc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if (epoch + 1) % 5 == 0 or epoch == EPOCHS - 1 or epoch == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} - Val BAcc: {val_bacc:.4f} - Best BAcc: {best_bacc:.4f}")
    model.load_state_dict(best_state)
    return model


def scores(y_true, y_pred):
    return {
        "acc": accuracy_score(y_true, y_pred),
        "bacc": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average="macro"),
    }


In [31]:
print("##" * 20)
print(" " * 20 +"RAW" + " " * 10)
print("##" * 20)

CONFIG = {
    "dataset": DATASET,
    "split_ratio": [0.8, 0.1, 0.1],
    "batch_size": 32,
    "seed": 100,
    "channel_mode": "raw",
    "return_ch_names": False,
}

train_loader, val_loader, test_loader = build_loaders(
    dataset=CONFIG["dataset"],
    split_ratio=CONFIG["split_ratio"],
    batch_size=CONFIG["batch_size"],
    seed=CONFIG["seed"],
    channel_mode=CONFIG["channel_mode"],
    return_ch_names=CONFIG["return_ch_names"]
    )


raw_model = train_eegnet(
    train_loader,
    val_loader,
    n_chans=train_loader.dataset[0][0].shape[0],
    n_times=train_loader.dataset[0][0].shape[1],
    n_classes=CLASSES[DATASET],
    class_weights=None,
)

s = scores(*evaluate(raw_model, test_loader))
print(f"end=> RAW Test Scores: acc={s['acc']:.4f}, bacc={s['bacc']:.4f}, f1={s['f1']:.4f}")

print("##" * 20)
print(" " * 20 + "MAPPED" + " " * 10)
print("##" * 20)

CONFIG = {
    "dataset": DATASET,
    "split_ratio": [0.8, 0.1, 0.1],
    "batch_size": 32,
    "seed": 100,
    "channel_mode": "mapped",
    "return_ch_names": False,
}

train_loader, val_loader, test_loader = build_loaders(
    dataset=CONFIG["dataset"],
    split_ratio=CONFIG["split_ratio"],
    batch_size=CONFIG["batch_size"],
    seed=CONFIG["seed"],
    channel_mode=CONFIG["channel_mode"],
    return_ch_names=CONFIG["return_ch_names"]
    )


mapped_model = train_eegnet(
    train_loader,
    val_loader,
    n_chans=train_loader.dataset[0][0].shape[0],
    n_times=train_loader.dataset[0][0].shape[1],
    n_classes=CLASSES[DATASET],
    class_weights=None,
)

s = scores(*evaluate(mapped_model, test_loader))
print(f"end=> MAPPED Test Scores: acc={s['acc']:.4f}, bacc={s['bacc']:.4f}, f1={s['f1']:.4f}")


########################################
                    RAW          
########################################
[LOADER:FACED] subjects=123 | source=ratio | train=99 | val=12 | test=12
Epoch 1/50 - Val BAcc: 0.1111 - Best BAcc: 0.1111
Epoch 5/50 - Val BAcc: 0.1860 - Best BAcc: 0.1952
Epoch 10/50 - Val BAcc: 0.2585 - Best BAcc: 0.3719
Epoch 15/50 - Val BAcc: 0.2230 - Best BAcc: 0.3920
Epoch 20/50 - Val BAcc: 0.1327 - Best BAcc: 0.3920
Epoch 25/50 - Val BAcc: 0.2068 - Best BAcc: 0.3920
Epoch 30/50 - Val BAcc: 0.2037 - Best BAcc: 0.3920
Epoch 35/50 - Val BAcc: 0.2168 - Best BAcc: 0.3920
Epoch 40/50 - Val BAcc: 0.2045 - Best BAcc: 0.3920
Epoch 45/50 - Val BAcc: 0.2106 - Best BAcc: 0.3920
Epoch 50/50 - Val BAcc: 0.2137 - Best BAcc: 0.3920
end=> RAW Test Scores: acc=0.3601, bacc=0.3472, f1=0.2761
########################################
                    MAPPED          
########################################
[LOADER:FACED] subjects=123 | source=ratio | train=99 | val=12 | test=12
Ep